# Context Dual Analysis Workflow`n
Converted from `context_dual_analysis_workflow.py` for interactive testing.

In [9]:
import os
import asyncio
import operator
from typing import TypedDict, List, Annotated, Dict, Any, Optional

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

from localsearch_engine_builder import build_search_engine


class DualContextAnalysisState(TypedDict):
    """
    Updated workflow state:
    user query -> local search -> MCP(DVIS) -> obligations -> capabilities -> gap mapping -> final report
    """

    messages: Annotated[List[BaseMessage], operator.add]
    user_query: str
    jurisdiction: str
    regulation_text: str
    dvis_document_content: str
    regulation_obligations_json: str
    dvis_capabilities_json: str
    gap_mapping_json: str
    final_report: str


INDEX_ROOT = os.path.join("..", "GraphRAG result", "data_privacy_csv_v2", "output")
LOCAL_SEARCH_PROMPT_PATH = os.path.join("system_prompt", "cust_local_search_system_prompt.txt")

REGULATION_TO_OBLIGATION_PROMPT_PATH = os.path.join(
    "system_prompt", "data privacy", "regulation to obligation.txt"
)
DVIS_TO_CAPABILITIES_PROMPT_PATH = os.path.join(
    "system_prompt", "data privacy", "DVIS to capabilities.txt"
)
GAP_ANALYSIS_PROMPT_PATH = os.path.join("system_prompt", "data privacy", "gap analysis.txt")
REPORT_FORMAT_PROMPT_PATH = os.path.join("system_prompt", "data privacy", "report format.txt")


print("Building Local Search engine...")
local_search_engine = build_search_engine(
    index_root=INDEX_ROOT,
    system_prompt_path=LOCAL_SEARCH_PROMPT_PATH,
    response_type="multiple paragraphs",
)
print("Local Search engine ready.")


def _load_prompt(path: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Prompt file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


REGULATION_TO_OBLIGATION_PROMPT = _load_prompt(REGULATION_TO_OBLIGATION_PROMPT_PATH)
DVIS_TO_CAPABILITIES_PROMPT = _load_prompt(DVIS_TO_CAPABILITIES_PROMPT_PATH)
GAP_ANALYSIS_PROMPT = _load_prompt(GAP_ANALYSIS_PROMPT_PATH)
REPORT_FORMAT_PROMPT = _load_prompt(REPORT_FORMAT_PROMPT_PATH)


def _extract_text_from_mcp_result(result: Any) -> str:
    """Normalize GitHub MCP get_file_contents output into plain text."""
    if isinstance(result, str):
        return result

    if isinstance(result, dict):
        if isinstance(result.get("text"), str):
            return result["text"]
        if isinstance(result.get("content"), str):
            return result["content"]
        return str(result)

    if isinstance(result, list):
        text_parts: List[str] = []
        for item in result:
            if isinstance(item, dict):
                if isinstance(item.get("text"), str):
                    text_parts.append(item["text"])
                elif isinstance(item.get("content"), str):
                    text_parts.append(item["content"])
            elif isinstance(item, str):
                text_parts.append(item)
        return "\n".join(part for part in text_parts if part.strip())

    return str(result)


async def initialize_mcp_github_client():
    """Initialize MCP GitHub client and return available tools."""
    github_token = os.environ.get("GITHUB_PERSONAL_ACCESS_TOKEN", "")
    if not github_token:
        print("Warning: GITHUB_PERSONAL_ACCESS_TOKEN not set")
        return None, []

    try:
        client = MultiServerMCPClient(
            {
                "github": {
                    "transport": "http",
                    "url": "https://api.githubcopilot.com/mcp/",
                    "headers": {"Authorization": f"Bearer {github_token}"},
                }
            }
        )
        tools = await client.get_tools()
        print(f"MCP GitHub Client initialized with {len(tools)} tools")
        return client, tools
    except Exception as e:
        print(f"Failed to initialize MCP client: {e}")
        return None, []


mcp_github_client = None
github_mcp_tools: List[Any] = []
github_read_file_tool = None


async def ensure_mcp_ready() -> None:
    global mcp_github_client, github_mcp_tools, github_read_file_tool
    if github_read_file_tool is not None:
        return

    mcp_github_client, github_mcp_tools = await initialize_mcp_github_client()
    if mcp_github_client is not None:
        github_read_file_tool = next(
            (t for t in github_mcp_tools if t.name == "get_file_contents"),
            None,
        )
    else:
        github_mcp_tools = []
        github_read_file_tool = None


@tool
async def local_search_tool(question: str) -> str:
    """Run GraphRAG local search for user's question."""
    result = await local_search_engine.search(question)
    return result.response


@tool
async def fetch_current_policy_rules(
    owner: str,
    repo: str,
    file_path: str,
    branch: str = "main",
) -> Dict[str, Any]:
    """
    Retrieve policy document content from GitHub via MCP.
    Returns raw document content for downstream analysis.
    """
    try:
        if not github_read_file_tool:
            raise RuntimeError("GitHub MCP tools not initialized.")

        result = await github_read_file_tool.ainvoke(
            {
                "owner": owner,
                "repo": repo,
                "path": file_path,
                "branch": branch,
            }
        )

        content = _extract_text_from_mcp_result(result)
        if not content.strip():
            raise RuntimeError("Empty content returned by get_file_contents")

        return {
            "document_content": content,
            "error": None,
        }
    except Exception as e:
        return {
            "document_content": "",
            "error": f"Failed to fetch document from GitHub: {e}",
        }


analysis_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# analysis_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

async def _run_prompt_chain_step(system_prompt: str, user_input: str) -> str:
    response = await analysis_llm.ainvoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_input),
        ]
    )
    return response.content if isinstance(response.content, str) else str(response.content)


async def local_search_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 1: Fetch Regulation Text (Local Search)")
    print("=" * 60)

    user_query = state.get("user_query", "")
    jurisdiction = state.get("jurisdiction", "")

    search_question = user_query
    if jurisdiction:
        search_question = f"What are the requirements for Transfer of personal data in {jurisdiction}?"

    regulation_text = await local_search_tool.ainvoke({"question": search_question})
    return {"regulation_text": regulation_text}


async def mcp_retrieve_document_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 2: Fetch DVIS.md via MCP")
    print("=" * 60)

    await ensure_mcp_ready()

    owner = "JJchan123"
    repo = "compliance_rule_configuration"
    file_path = "DataVisa-DVIS.md"
    branch = "main"

    result = await fetch_current_policy_rules.ainvoke(
        {
            "owner": owner,
            "repo": repo,
            "file_path": file_path,
            "branch": branch,
        }
    )

    if result.get("error"):
        error_msg = result["error"]
        return {
            "dvis_document_content": "",
            "messages": [AIMessage(content=f"MCP retrieval failed: {error_msg}")],
        }

    return {"dvis_document_content": result.get("document_content", "")}


async def regulation_to_obligation_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 3: Regulation to Obligations")
    print("=" * 60)

    jurisdiction = state.get("jurisdiction", "")
    regulation_text = state.get("regulation_text", "")

    user_input = (
        f"Jurisdiction: {jurisdiction}\n\n"
        f"Regulation Text:\n{regulation_text}\n"
    )
    obligations_json = await _run_prompt_chain_step(REGULATION_TO_OBLIGATION_PROMPT, user_input)
    return {"regulation_obligations_json": obligations_json}


async def dvis_to_capabilities_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 4: DVIS to Capabilities")
    print("=" * 60)

    dvis_document_content = state.get("dvis_document_content", "")
    user_input = f"DVIS Document Content:\n{dvis_document_content}\n"

    capabilities_json = await _run_prompt_chain_step(DVIS_TO_CAPABILITIES_PROMPT, user_input)
    return {"dvis_capabilities_json": capabilities_json}


async def gap_analysis_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 5: Gap Analysis Mapper")
    print("=" * 60)

    jurisdiction = state.get("jurisdiction", "")
    obligations_json = state.get("regulation_obligations_json", "")
    capabilities_json = state.get("dvis_capabilities_json", "")

    user_input = (
        f"Jurisdiction: {jurisdiction}\n\n"
        f"Regulation Obligations JSON:\n{obligations_json}\n\n"
        f"DVIS Capabilities JSON:\n{capabilities_json}\n"
    )

    gap_mapping_json = await _run_prompt_chain_step(GAP_ANALYSIS_PROMPT, user_input)
    return {"gap_mapping_json": gap_mapping_json}


async def report_formatter_node(state: DualContextAnalysisState) -> DualContextAnalysisState:
    print("\n" + "=" * 60)
    print("STEP 6: Final Report Formatter")
    print("=" * 60)

    gap_mapping_json = state.get("gap_mapping_json", "")
    user_input = f"Mapping Analysis Input:\n{gap_mapping_json}\n"

    final_report = await _run_prompt_chain_step(REPORT_FORMAT_PROMPT, user_input)
    return {
        "final_report": final_report,
        "messages": [AIMessage(content=final_report)],
    }


workflow = StateGraph(DualContextAnalysisState)
workflow.add_node("local_search", local_search_node)
workflow.add_node("mcp_retrieve_document", mcp_retrieve_document_node)
workflow.add_node("regulation_to_obligation", regulation_to_obligation_node)
workflow.add_node("dvis_to_capabilities", dvis_to_capabilities_node)
workflow.add_node("gap_analysis", gap_analysis_node)
workflow.add_node("report_formatter", report_formatter_node)

workflow.add_edge(START, "local_search")
workflow.add_edge("local_search", "mcp_retrieve_document")
workflow.add_edge("mcp_retrieve_document", "regulation_to_obligation")
workflow.add_edge("regulation_to_obligation", "dvis_to_capabilities")
workflow.add_edge("dvis_to_capabilities", "gap_analysis")
workflow.add_edge("gap_analysis", "report_formatter")
workflow.add_edge("report_formatter", END)

app = workflow.compile()
print("Updated gap-analysis workflow compiled successfully.")


async def run_workflow(
    user_question: str,
    jurisdiction: str = "",
) -> DualContextAnalysisState:
    """
    Execute updated workflow:
    local search -> MCP DVIS -> obligations -> capabilities -> gap mapping -> final report
    """
    initial_state: DualContextAnalysisState = {
        "messages": [HumanMessage(content=user_question)],
        "user_query": user_question,
        "jurisdiction": jurisdiction,
        "regulation_text": "",
        "dvis_document_content": "",
        "regulation_obligations_json": "",
        "dvis_capabilities_json": "",
        "gap_mapping_json": "",
        "final_report": "",
    }
    return await app.ainvoke(initial_state)


async def run_pvworkflow(
    user_question: str,
    jurisdiction: str = "",
) -> DualContextAnalysisState:
    """Alias for compatibility with existing notebook usage."""
    return await run_workflow(user_question=user_question, jurisdiction=jurisdiction)




Building Local Search engine...
Local Search engine ready.
Updated gap-analysis workflow compiled successfully.


In [15]:
# Notebook test example`n
# Optional: set env vars in notebook runtime`n
# import os`n
# os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = "..."`n
# os.environ["OPENAI_API_KEY"] = "..."`n
# os.environ["GEMINI_API_KEY"] = "..."`n
result = await run_pvworkflow(
    "Please do UAE vs DVIS gap analysis for transfer obligations",
    jurisdiction="UAE",
)
print(result["final_report"])


STEP 1: Fetch Regulation Text (Local Search)


Reached token limit - reverting to previous context state



STEP 2: Fetch DVIS.md via MCP

STEP 3: Regulation to Obligations

STEP 4: DVIS to Capabilities

STEP 5: Gap Analysis Mapper

STEP 6: Final Report Formatter
| Transfer obligation                                         | Relevant DVIS controls         | Coverage         | Note                                                                                      |
|-----------------------------------------------------------|-------------------------------|------------------|-------------------------------------------------------------------------------------------|
| obtain approval for cross-border transfer                  | DVIS.2.01, DVIS.2.02         | Partially covered | DVIS outlines the identification and implementation of data transfer conditions, but does not explicitly mention obtaining approval from a regulatory body. |
| ensure adequate protection in recipient jurisdiction        | DVIS.2.02                     | Partially covered | DVIS mentions implementing conditions for 

In [16]:
print(result["regulation_text"])  # 先看前 1000 字
print("\n")
# print(result["dvis_document_content"])
print("\n")
print(result["regulation_obligations_json"])
print("\n")
print(result["dvis_capabilities_json"])
print("\n")
# print(result["gap_mapping_json"])
# print(result["final_report"][:1000])


The transfer of personal data in the UAE is primarily governed by the "UAE-Concerning the Protection of Personal Data 2021" (Federal Decree by Law No. (45) of 2021), which establishes a comprehensive legal framework for data protection [Data: Entities (1, 5146, 5795)]. Cross-border transfer of personal data involves moving data across national borders, including from the UAE to other jurisdictions [Data: Entities (576); Relationships (1470, 13753, 13756)]. Such transfers are subject to specific conditions and approvals, and notably, they may increase risks for natural persons whose data is being moved, including the risk of unlawful use or disclosure of information [Data: Entities (576); Relationships (4563, 4589, 4590); Sources (61)].

### Conditions for Cross-Border Transfer with Adequate Protection (Article 22)

When a proper protection level is available in the recipient jurisdiction, personal data may be transferred outside the UAE in cases approved by the Bureau [Data: Entities (